# KTPM — bộ benchmark chung P1 / P2
Chọn Accelerator **None**, bật Internet, thêm dataset chứa `ktpm-p1-source.zip`.
Chạy `PROFILE = "smoke"` trước; dùng `"full"` để đo chính thức.
Khi có P2: thêm `ktpm-p2-source.zip`, bật dòng P2 bên dưới và chạy lại **tất cả cell**.
Hai pha dùng runner/config từ P1, cùng kernel CPU và database mới mỗi trial.
Không có kết quả Kaggle được điền sẵn. Xem docs/KAGGLE_CPU.md.

In [ ]:
import os, sys, json, hashlib, platform, subprocess, shutil, time, signal, zipfile, uuid
from pathlib import Path
from datetime import datetime, timezone
try:
    import psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'psutil==7.0.0'], check=True)
    import psutil
PROFILE = 'smoke'  # đổi thành 'full' để đo chính thức
PHASE_ARCHIVES = {'P1': 'ktpm-p1-source.zip'}
# PHASE_ARCHIVES['P2'] = 'ktpm-p2-source.zip'
CASE_FILTER = []  # [] = đủ 15 kịch bản; lọc chỉ để phát triển
ROOT = Path('/kaggle/working/ktpm-benchmark')
ROOT.mkdir(parents=True, exist_ok=True)
SESSION = uuid.uuid4().hex
OUT = ROOT / ('results-' + PROFILE + '-' + SESSION[:8])
OUT.mkdir()
SOURCES, HASHES = {}, {}
for phase, name in PHASE_ARCHIVES.items():
    matches = list(Path('/kaggle/input').rglob(name))
    if len(matches) != 1:
        raise RuntimeError('Cần đúng một archive: ' + name)
    source = ROOT / ('source-' + phase + '-' + SESSION[:8])
    source.mkdir()
    with zipfile.ZipFile(matches[0]) as archive:
        for entry in archive.infolist():
            destination = (source / entry.filename).resolve()
            if not destination.is_relative_to(source.resolve()) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise RuntimeError('Archive chứa đường dẫn không hợp lệ')
        archive.extractall(source)
    SOURCES[phase] = source
    HASHES[phase] = hashlib.sha256(matches[0].read_bytes()).hexdigest()
sys.path.insert(0, str(SOURCES['P1'] / 'scripts'))
import benchmark_suite as suite
import compare_benchmarks as comparison
# Reload avoids retaining a runner from an earlier execution of this notebook.
import importlib
import benchmark
importlib.reload(benchmark); importlib.reload(suite); importlib.reload(comparison)
CONFIG = json.loads((SOURCES['P1'] / 'scripts/benchmark_config.json').read_text(encoding='utf-8-sig'))
CASES = [c for c in suite.specifications(CONFIG, PROFILE) if not CASE_FILTER or c['id'] in CASE_FILTER]
if not CASES:
    raise RuntimeError('Không có kịch bản nào được chọn')
(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2))
print(PROFILE, len(CASES), 'trial / pha', 'harness:', suite.harness_hash())

## Build trước khi đo
Java, dependency và PostgreSQL portable tải trước. PostgreSQL chạy bằng user thường.
Mỗi pha hiện dùng cùng entrypoint `com.ktpm.LocalDemo`, Java heap 512 MiB và PostgreSQL 16.2.
Nếu P2 đổi cách triển khai, cần sửa adapter khởi chạy và giữ tổng ngân sách tài nguyên; không đổi workload.

In [ ]:
if shutil.which('java') is None:
    if os.geteuid() != 0:
        raise RuntimeError('Kernel thiếu JDK 17+')
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'], check=True)
if os.geteuid() == 0:
    if shutil.which('runuser') is None:
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'util-linux'], check=True)
    if subprocess.run(['id', 'ktpmbench'], capture_output=True).returncode:
        subprocess.run(['useradd', '--system', '--create-home', 'ktpmbench'], check=True)
    AS_USER = ['runuser', '-u', 'ktpmbench', '--']
else:
    AS_USER = []
M2 = ROOT / 'm2'
CLASSPATHS = {}
for phase, source in SOURCES.items():
    backend = source / 'backend'
    cp = ROOT / ('classpath-' + phase + '-' + SESSION[:8] + '.txt')
    subprocess.run(['bash', str(backend / 'mvnw'), '-f', str(backend / 'pom.xml'),
                    f'-Dmaven.repo.local={M2}', 'test-compile',
                    'org.apache.maven.plugins:maven-dependency-plugin:3.6.1:build-classpath',
                    f'-Dmdep.outputFile={cp}'], check=True)
    CLASSPATHS[phase] = os.pathsep.join([str(backend / 'target/test-classes'),
                                        str(backend / 'target/classes'), cp.read_text().strip()])
# The dedicated unprivileged server user must read sources, classes and Maven jars.
if AS_USER:
    subprocess.run(['chmod', '-R', 'a+rX', str(ROOT)], check=True)
JVM_OPTIONS = ['-Xms256m', '-Xmx512m', '-XX:ActiveProcessorCount=2']
def read_limit(name):
    path = Path('/sys/fs/cgroup') / name
    return path.read_text().strip() if path.exists() else None
ENVIRONMENT = dict(benchmarkSessionId=SESSION, platform=platform.platform(), python=sys.version,
                   logicalCpus=psutil.cpu_count(), affinity=psutil.Process().cpu_affinity(),
                   memoryBytes=psutil.virtual_memory().total,
                   java=subprocess.run(['java', '-version'], capture_output=True, text=True).stderr,
                   cpu=subprocess.run(['lscpu'], capture_output=True, text=True).stdout,
                   cpuMax=read_limit('cpu.max'), memoryMax=read_limit('memory.max'),
                   psutil=psutil.__version__, jvmOptions=JVM_OPTIONS, postgres='16.2',
                   topology='one backend + PostgreSQL + closed-loop client on same Kaggle CPU kernel',
                   selectedCases=CASE_FILTER,
                   trialMatrix=[{k:c[k] for k in ('id','concurrency','repeat')} for c in CASES])
(OUT / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2))
print(ENVIRONMENT['java'], ENVIRONMENT['logicalCpus'], 'logical CPUs')

In [ ]:
def stop_server(process):
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline:
        process.poll()
        try:
            os.killpg(process.pid, 0)
        except ProcessLookupError:
            return
        time.sleep(.25)
    # Only this notebook's process group, never all Java/PostgreSQL processes.
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=10)


def trial(phase, case):
    label = f"{case['id']}-c{case['concurrency']}-r{case['repeat']}"
    runtime = ROOT / ('runtime-' + SESSION[:8] + '-' + phase + '-' + label)
    import socket
    for port in (8080, 55432):
        with socket.socket() as probe:
            try:
                probe.bind(('127.0.0.1', port))
            except OSError:
                raise RuntimeError(f'Port {port} đang bị dùng; không tái sử dụng server cũ')
    runtime.mkdir()  # never reuse an old trial/database
    pg_data = runtime / 'postgres'
    if AS_USER:
        subprocess.run(['chown', '-R', 'ktpmbench:ktpmbench', str(runtime)], check=True)
    process = None
    with (runtime / 'backend.log').open('w') as log:
        try:
            env = os.environ.copy()
            import secrets
            env['JWT_SECRET'] = secrets.token_urlsafe(48)
            command = AS_USER + ['java'] + JVM_OPTIONS + [f'-Dktpm.local.data={pg_data}',
                       '-cp', CLASSPATHS[phase], 'com.ktpm.LocalDemo']
            process = subprocess.Popen(command, cwd=SOURCES[phase] / 'backend', env=env,
                                       stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            deadline = time.monotonic() + 120
            while benchmark.api('http://localhost:8080', '/actuator/health', timeout=2)[0] != 200:
                if process.poll() is not None or time.monotonic() > deadline:
                    raise RuntimeError('Backend chưa sẵn sàng; xem ' + str(runtime / 'backend.log'))
                time.sleep(1)
            candidates = [psutil.Process(process.pid)] + psutil.Process(process.pid).children(recursive=True)
            java_pid = next(p.pid for p in candidates if p.name().startswith('java'))
            database_pid = int((pg_data / 'postmaster.pid').read_text().splitlines()[0])
            return suite.run_case('http://localhost:8080', case, OUT / phase / label, phase,
                                  [java_pid], [database_pid], require_empty=True,
                                  experiment=dict(environment=ENVIRONMENT, sourceSha256=HASHES[phase]))
        finally:
            stop_server(process)
            pid_file = pg_data / 'postmaster.pid'
            if pid_file.exists() and psutil.pid_exists(int(pid_file.read_text().splitlines()[0])):
                raise RuntimeError('PostgreSQL của trial chưa dừng; giữ data và dừng pipeline')
            if pg_data.exists():
                assert pg_data.resolve().parent == runtime.resolve() and runtime.resolve().parent == ROOT.resolve()
                shutil.rmtree(pg_data)  # only this trial's stopped database, not source/results
    # Tokens only live in Python memory. Logs/database are excluded from output ZIP.

## Đo và xuất kết quả
Full: 108 trial mỗi pha, ba lần lặp. Soak 300 giây/lần; build, seed và audit ngoài khoảng đo.
P1/P2 chạy xen kẽ, đảo thứ tự ở lần lặp chẵn. Không chạy notebook benchmark khác cùng lúc.
Các file report, samples và timeline cho phép phân tích từng endpoint, mức tải và dữ liệu sai.
Nếu trial lỗi, sửa nguyên nhân rồi chạy lại; không ghép một tập kết quả thiếu trial.

In [ ]:
reports = []
for case in CASES:
    phases = list(PHASE_ARCHIVES)
    if case['repeat'] % 2 == 0:
        phases.reverse()
    for phase in phases:
        print('Trial:', phase, case['id'], case['concurrency'], case['repeat'], flush=True)
        reports.append(trial(phase, case))
if 'P2' in PHASE_ARCHIVES:
    rows = comparison.write_comparison(OUT / 'P1', OUT / 'P2', OUT / 'comparison')
    print('Comparison:', OUT / 'comparison/comparison.csv')
archive_path = OUT.with_suffix('.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.suffix in ('.json', '.jsonl', '.csv', '.md'):
            archive.write(path, str(path.relative_to(OUT)))
print('Download:', archive_path)

## Đọc kết quả
`successfulRps`: request thành công và JSON đúng; 409 bid được báo riêng, không tính thành công.
Đọc p95/p99 cả tất cả request lẫn request thành công, lỗi và kết quả audit cùng nhau.
Scheduler lag là cận trên quan sát qua API, gồm thời gian monitor, không phải thời điểm nội bộ chính xác.
`comparison.csv`: median/min/max ba trial, % cải thiện; số dương nghĩa là tốt hơn.
Nếu correctness sai/không xác định hoặc đang dùng smoke: không xuất % cải thiện chính thức.
Comparator từ chối khác kernel, harness, workload, database không mới hoặc thiếu lần lặp.
Giữ ZIP P1/runner/config làm baseline cho P2; nếu đổi workload phải đo lại cả P1/P2.
Khi cải tiến nhiều instance, ghi topology và CPU/RAM tổng; sửa adapter đo tất cả PID và ngân sách trước khi kết luận.